# vllm-chat — Jupyter Notebook Chat Client for a vLLM-served Model

Connect to a running vLLM server and chat with your fine-tuned model directly from this notebook.

**Quick start:**
1. Edit the **Configuration** cell below (set `VLLM_URL`, `MODEL`, `SYSTEM_PROMPT`, etc.)
2. Run all cells top-to-bottom
3. Use `ask("your message")` in any cell to chat — multi-turn history is kept automatically

In [ ]:
from __future__ import annotations

import argparse
import json
import os
import sys
import textwrap
import urllib.error
import urllib.request
from datetime import datetime
from pathlib import Path

## ANSI colour helpers (gracefully disabled on non-TTY / Windows without VT)

In [ ]:
def _supports_colour() -> bool:
    if not sys.stdout.isatty():
        return False
    if os.name == "nt":
        try:
            import ctypes
            k32 = ctypes.windll.kernel32
            k32.SetConsoleMode(k32.GetStdHandle(-11), 7)
            return True
        except Exception:
            return False
    return True

USE_COLOUR = _supports_colour()

In [ ]:
def _c(code: str, text: str) -> str:
    return f"\033[{code}m{text}\033[0m" if USE_COLOUR else text

In [ ]:
def cyan(t):    return _c("36", t)
def green(t):   return _c("32", t)
def yellow(t):  return _c("33", t)
def red(t):     return _c("31", t)
def bold(t):    return _c("1",  t)
def dim(t):     return _c("2",  t)
def magenta(t): return _c("35", t)

## vLLM API helpers

In [ ]:
def _get_models(base_url: str, api_key: str = "") -> list[str]:
    """Return available model names from the vLLM server."""
    url = f"{base_url.rstrip('/')}/v1/models"
    headers = {"Accept": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=10) as r:
            data = json.loads(r.read().decode())
            return [m["id"] for m in data.get("data", [])]
    except Exception:
        return []

In [ ]:
def _chat_completion(
    base_url: str,
    model: str,
    messages: list[dict],
    temperature: float,
    max_tokens: int,
    stream: bool,
    api_key: str = "",
) -> str:
    """
    Call POST /v1/chat/completions.
    If stream=True, prints tokens as they arrive and returns full text.
    If stream=False, returns full response text.
    """
    url = f"{base_url.rstrip('/')}/v1/chat/completions"
    payload = json.dumps({
        "model":       model,
        "messages":    messages,
        "temperature": temperature,
        "max_tokens":  max_tokens,
        "stream":      stream,
    }).encode()

    headers = {
        "Content-Type": "application/json",
        "Accept":       "text/event-stream" if stream else "application/json",
    }
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    req = urllib.request.Request(url, data=payload, headers=headers, method="POST")

    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            if not stream:
                data = json.loads(resp.read().decode())
                return data["choices"][0]["message"]["content"]
            else:
                # Stream: print tokens as they arrive
                full_text = ""
                print(green(bold("assistant")) + ": ", end="", flush=True)
                for raw_line in resp:
                    line = raw_line.decode("utf-8").strip()
                    if not line or not line.startswith("data:"):
                        continue
                    chunk_str = line[len("data:"):].strip()
                    if chunk_str == "[DONE]":
                        break
                    try:
                        chunk = json.loads(chunk_str)
                        delta = chunk["choices"][0]["delta"].get("content", "")
                        if delta:
                            print(delta, end="", flush=True)
                            full_text += delta
                    except json.JSONDecodeError:
                        continue
                print()  # newline after stream
                return full_text

    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="ignore")
        raise RuntimeError(f"HTTP {e.code}: {body[:300]}")
    except urllib.error.URLError as e:
        raise RuntimeError(f"Cannot reach {url}: {e.reason}")

## Text formatting

In [ ]:
def _wrap_text(text: str, width: int = 100, indent: str = "  ") -> str:
    """Wrap text preserving code blocks and indentation."""
    lines = text.split("\n")
    result = []
    in_code = False

    for line in lines:
        if line.startswith("```"):
            in_code = not in_code
            result.append(line)
            continue
        if in_code:
            result.append(line)
        else:
            if len(line) > width:
                wrapped = textwrap.fill(line, width=width, subsequent_indent=indent)
                result.append(wrapped)
            else:
                result.append(line)

    return "\n".join(result)

In [ ]:
def _print_assistant_response(text: str):
    """Print the assistant response with syntax-aware formatting."""
    lines = text.split("\n")
    in_code = False
    code_lang = ""

    for line in lines:
        if line.startswith("```"):
            if not in_code:
                in_code = True
                code_lang = line[3:].strip()
                print(dim(f"  ┌─ {code_lang or 'code'} " + "─" * max(0, 30 - len(code_lang))))
            else:
                in_code = False
                print(dim("  └" + "─" * 33))
        elif in_code:
            print("  " + yellow(line))
        else:
            if line.startswith("**") and line.endswith("**"):
                print(bold(line))
            elif line.startswith("## ") or line.startswith("# "):
                print(bold(cyan(line)))
            else:
                print(line)

## Conversation save / load

In [ ]:
SAVE_DIR = Path("chat_history")

def _save_conversation(messages: list[dict], system: str) -> str:
    SAVE_DIR.mkdir(exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = SAVE_DIR / f"conversation_{ts}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"system": system, "messages": messages}, f, indent=2, ensure_ascii=False)
    return str(path)

In [ ]:
def _load_conversation(path: str) -> tuple[list[dict], str]:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return data.get("messages", []), data.get("system", "")

## ⚙️ Configuration & Session

In [ ]:
# ── Configuration — edit these variables before running ───────────────────────

VLLM_URL         = "http://localhost:8000"   # vLLM server base URL
MODEL            = ""                         # leave "" to auto-detect from server
TEMPERATURE      = 0.4                        # 0.0 = deterministic, higher = creative
MAX_TOKENS       = 2048                       # max tokens per response
ENABLE_STREAMING = False                      # True = print tokens live as generated
API_KEY          = os.environ.get("VLLM_API_KEY", "")   # set if server requires auth

SYSTEM_PROMPT = (
    "You are an expert Java security engineer specialising in vulnerability remediation. "
    "When asked to fix vulnerable Java code, produce a corrected, secure version inside "
    "a ```java ... ``` code block. Be concise and precise."
)

# ── Session state ──────────────────────────────────────────────────────────────
_history: list[dict] = []

_model = MODEL
if not _model:
    _detected = _get_models(VLLM_URL, API_KEY)
    _model = _detected[0] if _detected else "default"
    if _detected:
        print(f"Auto-detected model: {_model}")
    else:
        print(f"[WARN] No model detected at {VLLM_URL} — is the vLLM server running?")


def ask(message: str, *, stream: bool = ENABLE_STREAMING, remember: bool = True) -> str:
    """
    Send a message to the model and return the response.

    Args:
        message:  Your prompt or code snippet.
        stream:   Print tokens as they arrive (default: ENABLE_STREAMING).
        remember: Keep this exchange in conversation history (default True).

    Returns:
        The model's full response as a string.
    """
    _history.append({"role": "user", "content": message})
    msgs = []
    if SYSTEM_PROMPT:
        msgs.append({"role": "system", "content": SYSTEM_PROMPT})
    msgs.extend(_history)

    try:
        response = _chat_completion(
            base_url    = VLLM_URL,
            model       = _model,
            messages    = msgs,
            temperature = TEMPERATURE,
            max_tokens  = MAX_TOKENS,
            stream      = stream,
            api_key     = API_KEY,
        )
    except RuntimeError as e:
        _history.pop()
        print(red(f"[ERROR] {e}"))
        return ""

    if not stream:
        _print_assistant_response(response)

    if remember:
        _history.append({"role": "assistant", "content": response})
    else:
        _history.pop()

    return response


def clear_history():
    """Clear conversation history (keeps system prompt and config)."""
    _history.clear()
    print("Conversation history cleared.")


def show_history():
    """Print a summary of the current conversation history."""
    if not _history:
        print("No history yet.")
        return
    print(f"\n{'─'*55}")
    for i, m in enumerate(_history):
        preview = m["content"][:120].replace("\n", " ")
        print(f"  [{i}] {m['role'].upper()}: {preview}{'...' if len(m['content']) > 120 else ''}")
    print(f"{'─'*55}\n")


def save_session() -> str:
    """Save conversation history to a JSON file in chat_history/."""
    saved = _save_conversation(_history, SYSTEM_PROMPT)
    print(f"Session saved to: {saved}")
    return saved


def load_session(path: str):
    """Restore a previously saved conversation from a JSON file."""
    msgs, _ = _load_conversation(path)
    _history.clear()
    _history.extend(msgs)
    print(f"Loaded {len(msgs)} messages from {path}")


print(f"\n{'━'*55}")
print(f"  vLLM Notebook Chat  —  ready")
print(f"{'━'*55}")
print(f"  Server : {VLLM_URL}")
print(f"  Model  : {_model}")
print(f"  Temp   : {TEMPERATURE}   Max tokens: {MAX_TOKENS}   Stream: {ENABLE_STREAMING}")
print(f"{'━'*55}")
print(f"  ask('your message')   — send a message")
print(f"  clear_history()       — start a new conversation")
print(f"  show_history()        — see all turns so far")
print(f"  save_session()        — save to chat_history/")
print(f"  load_session(path)    — restore a saved session")
print(f"{'━'*55}")

## 💬 Chat — Edit the cell below and run it to send a message

In [ ]:
# Edit this message and run the cell to chat.
# Run it multiple times — conversation history is kept between runs.

response = ask("""## Security Vulnerability Report

**CVE ID:** CVE-2021-EXAMPLE
**Vulnerability Type:** SQL Injection

## Vulnerable Java Code

```java
public List<User> getUser(String username) {
    String query = "SELECT * FROM users WHERE username = '" + username + "'";
    return jdbcTemplate.query(query, new UserRowMapper());
}
```

Please provide the fixed, secure version of this code.
""")

In [ ]:
# ── Session utilities — uncomment and run as needed ───────────────────────────

# show_history()     # print all turns so far

# clear_history()    # start fresh (keeps system prompt and config)

# save_session()     # save conversation to chat_history/ folder

# load_session("chat_history/conversation_20260611_120000.json")  # restore